# AraBERT and Modern Arabic NLP Modules Exercises

## Learning Goal
- Understand what AraBERT is and how it fits Arabic NLP.
- Learn how to use Hugging Face Transformers for Arabic masked-language modeling.
- Learn how similar modern modules such as CAMeLBERT, MARBERT, ARBERT, BGE-M3, and SentenceTransformers-style models fit real AI engineering.
- Practice Arabic preprocessing, embeddings, semantic similarity, and production-style preparation for retrieval.

### Background & links about the library
- AraBERT model card: https://huggingface.co/aubmindlab/bert-base-arabert
- AraBERT v0.2: https://huggingface.co/aubmindlab/bert-base-arabertv02
- AraBERT paper: https://huggingface.co/papers/2003.00104
- AUB MIND LAB models: https://huggingface.co/aubmindlab
- CAMeLBERT collection: https://huggingface.co/collections/CAMeL-Lab/camelbert
- ARBERT and MARBERT collection: https://huggingface.co/collections/UBC-NLP/arbert-and-marbert
- BGE-M3: https://huggingface.co/BAAI/bge-m3
- Sentence Transformers: https://www.sbert.net/
- Transformers docs: https://huggingface.co/docs/transformers

### Why it matters in real AI engineering
Arabic NLP is difficult because of morphology, dialects, diacritics, orthographic variation, OCR noise, and classical-vs-modern style differences. In real systems, Arabic models support classification, semantic search, RAG retrieval, reranking, evaluation, duplicate detection, and document understanding.

## Mental Model

AraBERT and similar modules are **Arabic text understanding engines**.

```text
Arabic text
    ↓
cleaning / normalization
    ↓
tokenizer
    ↓
transformer model
    ↓
hidden states / logits / embeddings
    ↓
task output
```

### Model family mental map

```text
AraBERT / CAMeLBERT / MARBERT / ARBERT
    → Arabic language understanding, classification, fill-mask, NER experiments

SentenceTransformers / BGE-M3 / Arabic-tuned embedding models
    → embeddings, semantic search, retrieval, vector DBs, RAG

AraGPT2 / Jais / Qwen-style LLMs
    → generation and chat

Rerankers / cross-encoders
    → more precise query-document scoring
```

### How this concept fits production systems

```text
Arabic documents → cleaning → chunking → embedding model → vector DB → retrieval → reranking → LLM → answer with citations
```

AraBERT-like models are often not the final generator. They are usually understanding, classification, embedding, or ranking components.

## Background Explanation

### Core concepts

#### Transformer
A transformer learns contextual representations of text. For Arabic, context is crucial because words such as `عين` can mean different things depending on surrounding words.

#### Tokenizer
A tokenizer converts text into token IDs. Arabic tokenization is hard because Arabic words can contain prefixes, suffixes, clitics, and attached pronouns.

#### Masked Language Modeling
BERT-style models predict missing tokens:

```text
الذكاء الاصطناعي يساعد في [MASK] النصوص.
```

#### Embeddings
Embeddings are vectors representing meaning. They are used for semantic search, clustering, retrieval, duplicate detection, and vector databases.

#### Fine-tuning
Fine-tuning adapts a pretrained model to a specific task such as sentiment classification, topic classification, named entity recognition, or question answering.

### Model selection

| Model family | Best use |
|---|---|
| AraBERT | MSA understanding, classification, fill-mask |
| CAMeLBERT | MSA, dialectal Arabic, classical/mixed variants depending on checkpoint |
| MARBERT | social media and dialect-heavy Arabic |
| ARBERT | broad Arabic understanding |
| BGE-M3 | multilingual embeddings and retrieval |
| Arabic-tuned BGE variants | Arabic retrieval/RAG after validation |
| AraGPT2 / Jais / Qwen | generation/chat, not embeddings by default |

## Setup

This notebook has two modes:

### Lightweight learning mode
Uses preprocessing and fallback toy vectors. Works without GPU.

### Real model mode
Uses Hugging Face Transformers and Sentence Transformers. Requires internet the first time models are downloaded.

Recommended install:

```bash
pip install transformers torch sentence-transformers numpy pandas
```

In [1]:
# Install required packages if missing.
import sys, subprocess, importlib.util
packages = {
    "transformers": "transformers",
    "torch": "torch",
    "sentence_transformers": "sentence-transformers",
    "numpy": "numpy",
    "pandas": "pandas",
}
missing = [pip for module, pip in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [2]:
import re
import numpy as np
import pandas as pd
print("Basic libraries imported successfully.")

Basic libraries imported successfully.


## Minimal Working Example

We will start with:

1. A real AraBERT fill-mask example using Hugging Face Transformers.
2. A fallback Arabic normalization example that always runs without model download.

The AraBERT example requires internet access the first time the model is downloaded.

In [3]:
# Minimal Working Example A: AraBERT fill-mask
from transformers import pipeline

model_name = "aubmindlab/bert-base-arabertv02"
try:
    fill_mask = pipeline("fill-mask", model=model_name, tokenizer=model_name)
    sentence = "الذكاء الاصطناعي يساعد في [MASK] النصوص العربية."
    predictions = fill_mask(sentence)
    for pred in predictions[:5]:
        print(pred["sequence"], "=>", round(pred["score"], 4))
except Exception as e:
    print("Could not run AraBERT fill-mask example.")
    print("Reason:", type(e).__name__, str(e)[:300])
    print("You can continue with the rest of the notebook.")

c:\AI\environments\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\AI\environments\rag\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\AI\cache\huggingface\hub\models--aubmindlab--bert-base-arabertv02. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this artic

الذكاء الاصطناعي يساعد في ترجمة النصوص العربية. => 0.4159
الذكاء الاصطناعي يساعد في قراءة النصوص العربية. => 0.1201
الذكاء الاصطناعي يساعد في كتابة النصوص العربية. => 0.0733
الذكاء الاصطناعي يساعد في معالجة النصوص العربية. => 0.0383
الذكاء الاصطناعي يساعد في فهم النصوص العربية. => 0.0321


In [4]:
# Minimal Working Example B: simple Arabic normalization

def normalize_arabic(text: str) -> str:
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

examples = [
    "إِنَّ الذكاءَ الاصطناعيَّ مهمٌّ.",
    "الذكاء الاصطناعي مهم",
    "آثارُ النماذجِ اللغويةِ كبيرةٌ.",
]
for text in examples:
    print("Original:   ", text)
    print("Normalized: ", normalize_arabic(text))
    print("-" * 60)

Original:    إِنَّ الذكاءَ الاصطناعيَّ مهمٌّ.
Normalized:  ان الذكاء الاصطناعي مهم.
------------------------------------------------------------
Original:    الذكاء الاصطناعي مهم
Normalized:  الذكاء الاصطناعي مهم
------------------------------------------------------------
Original:    آثارُ النماذجِ اللغويةِ كبيرةٌ.
Normalized:  اثار النماذج اللغويه كبيره.
------------------------------------------------------------


### Short explanation

The fill-mask example shows how AraBERT predicts contextually likely tokens. The normalization example shows a production truth: Arabic model quality depends heavily on text cleaning, chunking, metadata, and domain evaluation.

## Exercise 1

### Goal
Build a small Arabic preprocessing function for model input.

Your function should:

- remove tatweel `ـ`
- remove Arabic diacritics
- normalize Alef forms to `ا`
- normalize `ى` to `ي`
- collapse extra whitespace

### TODO section
Complete the function below.

### Questions to think about
- Why can diacritics create matching problems in search?
- Should we always remove diacritics for classical Arabic?
- What could go wrong if we normalize too aggressively?
- Should the original raw text be stored too?

In [ ]:
# Exercise 1 TODO
import re

# \u064B  # Tanween Fath
# \u064C  # Tanween Damm
# \u064D  # Tanween Kasr
# \u064E  # Fathah
# \u064F  # Dhammah
# \u0650  # Kasrah
# \u0651  # Shaddah
# \u0652  # Sukun
# \u0670  # Dagger Alif
# \u0622  # Alif Maddah
# \u0623  # Alif Hamza Above
# \u0625  # Alif Hamza Below
# \u0649  # Alif Maqsurah
# \u0627  # Plain Alif
# \u064A  # Regular Ya
# \s     # Whitespace (Spaces, tabs, newlines)

def clean_arabic_text(text: str) -> str:
    # TODO 1: remove tatweel character ـ
    text = re.sub("ـ", "", text)


    # TODO 2: remove Arabic diacritics
    diacritics = re.compile(r'[\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652\u0670]')

    text = diacritics.sub("", text)

    # TODO 3: normalize Alef forms إ أ آ ا to ا
    alef_normalize = re.compile(r'[\u0622\u0623\u0625]')
    
    text = alef_normalize.sub("\u0627", text)

    # TODO 4: normalize ى to ي
    alef_maqsora_normalize = re.compile(r'[\u0649]')

    text = alef_maqsora_normalize.sub("ي", text)

    # TODO 5: collapse repeated whitespace
    space_pattern = re.compile(r'[\s]+')
    text = space_pattern.sub(' ', text)

    return text

sample_text = "إِنَّــــ الذكاءَ   الاصطناعيَّ نافعٌ في معالجةِ النُّصوصِ العربية."
print(clean_arabic_text(sample_text))

ان الذكاء الاصطناعي نافع في معالجة النصوص العربية.


## Solution 1

In [ ]:
# Solution 1
import re

def clean_arabic_text(text: str) -> str:
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

sample_text = "إِنَّــــ الذكاءَ   الاصطناعيَّ نافعٌ في معالجةِ النُّصوصِ العربية."
print("Original: ", sample_text)
print("Cleaned:  ", clean_arabic_text(sample_text))

## Exercise 2

### Slightly harder
Build a small Arabic semantic similarity example.

In production, you would use an embedding model such as BGE-M3, multilingual-e5, Arabic sentence-transformer models, or Arabic-tuned BGE variants.

### Goal
Create a function that:

1. takes a user query
2. embeds the query
3. embeds candidate Arabic passages
4. computes cosine similarity
5. returns the most similar passages

### Questions to think about
- Why are embeddings useful for RAG?
- Why is AraBERT fill-mask not enough for retrieval?
- Why should query and documents use the same embedding model?
- How would you store these vectors in Qdrant?

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch

model_name = "BAAI/bge-m3"

# Exercise 2 TODO
arabic_passages = [
    "الذكاء الاصطناعي يساعد في تحليل النصوص العربية.",
    "قواعد البيانات المتجهية تستخدم للبحث الدلالي.",
    "تطبيقات الويب تحتاج إلى واجهات برمجية مثل FastAPI.",
    "تجزئة الكتب العربية مهمة في أنظمة الاسترجاع المعزز بالتوليد.",
]
query = "كيف نبحث دلاليا في النصوص العربية؟"

# TODO:
# 1. Try loading SentenceTransformer("BAAI/bge-m3").
model = SentenceTransformer(model_name)

# 2. Encode the query and passages.
query_embedding = model.encode(query)
passage_embeddings = model.encode(arabic_passages)
cosine_similarities = model.similarity(query_embedding, passage_embeddings)

print(f'query_embedding{query_embedding}')
print(f'passage_embeddings{passage_embeddings[2]}')
print(f'cosine_similarities{cosine_similarities}')

# 3. Compute cosine similarity.
max_index = torch.argmax(cosine_similarities).item()

# 4. Return passages sorted by similarity.
arabic_passages[max_index]
# 5. If model loading fails, implement a fallback keyword-based toy vectorizer.

query_embedding[-0.00042402 -0.00582784 -0.00127649 ...  0.01534142  0.00433159
  0.02255349]
passage_embeddings[-0.0329924  -0.00488655 -0.04890068 ...  0.00385606  0.01826535
  0.01468385]
cosine_similaritiestensor([[0.6055, 0.5424, 0.3895, 0.4685]])


'الذكاء الاصطناعي يساعد في تحليل النصوص العربية.'

## Solution 2

In [50]:
# Solution 2
from typing import List, Tuple
import numpy as np

arabic_passages = [
    "الذكاء الاصطناعي يساعد في تحليل النصوص العربية.",
    "قواعد البيانات المتجهية تستخدم للبحث الدلالي.",
    "تطبيقات الويب تحتاج إلى واجهات برمجية مثل FastAPI.",
    "تجزئة الكتب العربية مهمة في أنظمة الاسترجاع المعزز بالتوليد.",
]
query = "كيف نبحث دلاليا في النصوص العربية؟"

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)

def fallback_keyword_vector(text: str) -> np.ndarray:
    vocabulary = ["بحث", "دلالي", "النصوص", "العربية", "قواعد", "المتجهية", "تجزئة", "استرجاع", "FastAPI"]
    cleaned = clean_arabic_text(text)
    return np.array([1.0 if term in cleaned else 0.0 for term in vocabulary], dtype=np.float32)

def rank_passages(query: str, passages: List[str]) -> List[Tuple[float, str]]:
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("BAAI/bge-m3")
        query_embedding = model.encode(query, normalize_embeddings=True)
        passage_embeddings = model.encode(passages, normalize_embeddings=True)
        scores = [cosine_similarity(query_embedding, emb) for emb in passage_embeddings]
    except Exception as e:
        print("Could not load real embedding model. Using fallback toy vectorizer.")
        print("Reason:", type(e).__name__, str(e)[:200])
        query_embedding = fallback_keyword_vector(query)
        passage_embeddings = [fallback_keyword_vector(p) for p in passages]
        scores = [cosine_similarity(query_embedding, emb) for emb in passage_embeddings]
    return sorted(zip(scores, passages), key=lambda x: x[0], reverse=True)

for score, passage in rank_passages(query, arabic_passages):
    print(round(score, 4), "=>", passage)

0.6055 => الذكاء الاصطناعي يساعد في تحليل النصوص العربية.
0.5424 => قواعد البيانات المتجهية تستخدم للبحث الدلالي.
0.4685 => تجزئة الكتب العربية مهمة في أنظمة الاسترجاع المعزز بالتوليد.
0.3895 => تطبيقات الويب تحتاج إلى واجهات برمجية مثل FastAPI.


## Common Mistakes

### Mistake 1: Treating all Arabic models as the same
AraBERT, MARBERT, CAMeLBERT, ARBERT, BGE-M3, and Arabic-tuned embedding models serve different purposes.

### Mistake 2: Using fill-mask models as retrieval models
A fill-mask model predicts missing words. A retrieval embedding model compares semantic similarity.

### Mistake 3: Indexing documents with one model and querying with another
Use the same embedding model for stored document vectors and query vectors.

### Mistake 4: Ignoring Arabic normalization
Forms such as `إسلام`, `اسلام`, and `إِسْلام` may need normalization depending on the task.

### Mistake 5: Over-normalizing classical Arabic
For classical Arabic, exact spelling and diacritics may sometimes matter. Store raw text and normalized text.

### Mistake 6: Forgetting model max length
Do not pass whole books directly to BERT-style encoders. Chunk first.

### Mistake 7: Not evaluating on your own domain
Modern Arabic web text, dialect text, OCR text, and classical books behave differently.

In [59]:
# Debugging example: raw vs normalized matching
query = "اسلام"
documents = ["الإسلام دين عظيم.", "إِسْلام", "هذا نص عن الذكاء الاصطناعي."]

print("Raw matching:")
for doc in documents:
    print(query in doc, "=>", doc)

print("Normalized matching:")
normalized_query = clean_arabic_text(query)
for doc in documents:
    normalized_doc = clean_arabic_text(doc)
    print(normalized_query in normalized_doc, "=>", normalized_doc)

Raw matching:
False => الإسلام دين عظيم.
False => إِسْلام
False => هذا نص عن الذكاء الاصطناعي.
Normalized matching:
True => الاسلام دين عظيم.
True => اسلام
False => هذا نص عن الذكاء الاصطناعي.


## Real AI Engineering Usage

### RAG
Arabic RAG usually needs cleaning, chunking, embeddings, vector DB storage, retrieval, reranking, LLM generation, and citations.

### APIs
A FastAPI service might expose `/normalize`, `/embed`, `/classify`, `/search`, and `/rerank`.

### ML pipelines
Arabic models appear in sentiment classification, topic classification, NER, duplicate detection, semantic clustering, and relevance scoring.

### Ingestion
Arabic NLP modules help with text normalization, language/dialect detection, chunk quality scoring, metadata enrichment, and OCR cleanup.

### Evaluation
Embedding models can support semantic similarity checks, retrieval evaluation, clustering of failures, and weak-signal grading. Human review remains important for sensitive domains.

### Vector DBs
Embedding models produce vectors; Qdrant or another vector DB stores them with payload metadata.

Example payload:

```python
{
    "chunk_id": "book-001-page-012-chunk-03",
    "text": "...",
    "normalized_text": "...",
    "book": "كتاب عربي",
    "author": "المؤلف",
    "page_start": 12,
    "section": "باب العلم"
}
```

## Final Mini Exercise

### Small production-style task
Build a mini Arabic retrieval preparation module.

Create:

```python
prepare_arabic_chunks_for_retrieval(chunks)
```

Each input chunk should include `chunk_id`, `text`, `source`, and `page`.

Return records with:

- `chunk_id`
- `raw_text`
- `normalized_text`
- `source`
- `page`
- `text_length`
- `ready_for_embedding`

In [ ]:
import copy
# Final Mini Exercise TODO
chunks = [
    {"chunk_id": "arabic-book-001", "text": "إِنَّ الذكاءَ الاصطناعيَّ يساعدُ في تحليلِ النصوصِ.", "source": "arabic_ai_book.md", "page": 1},
    {"chunk_id": "arabic-book-002", "text": "قواعدُ البياناتِ المتجهيةُ مفيدةٌ في البحثِ الدلالي.", "source": "arabic_ai_book.md", "page": 2},
]

# Original is updated - passing parameter
def prepare_arabic_chunks_for_retrieval(chunks):
    new_chunks = copy.deepcopy(chunks)
    # TODO:
    # - keep raw_text
    # - create normalized_text using clean_arabic_text
    # - keep metadata
    # - compute text_length
    # - set ready_for_embedding to True if normalized_text length > 20

    for chunk in new_chunks:
        chunk["text"] = normalize_arabic(chunk["text"])
        chunk["length"] = len(chunk["text"])
        # chunk["ready_for_embedding"] = True if chunk["length"] > 20 else False
        if chunk["length"] > 20:
            chunk["ready_for_embedding"] = True
        else:
            chunk["ready_for_embedding"] = False

    return new_chunks

updated_chunks = prepare_arabic_chunks_for_retrieval(chunks=chunks)
print(updated_chunks)
print(chunks)


[{'chunk_id': 'arabic-book-001', 'text': 'ان الذكاء الاصطناعي يساعد في تحليل النصوص.', 'source': 'arabic_ai_book.md', 'page': 1, 'length': 42, 'ready_for_embedding': True}, {'chunk_id': 'arabic-book-002', 'text': 'قواعد البيانات المتجهيه مفيده في البحث الدلالي.', 'source': 'arabic_ai_book.md', 'page': 2, 'length': 47, 'ready_for_embedding': True}]
[{'chunk_id': 'arabic-book-001', 'text': 'إِنَّ الذكاءَ الاصطناعيَّ يساعدُ في تحليلِ النصوصِ.', 'source': 'arabic_ai_book.md', 'page': 1}, {'chunk_id': 'arabic-book-002', 'text': 'قواعدُ البياناتِ المتجهيةُ مفيدةٌ في البحثِ الدلالي.', 'source': 'arabic_ai_book.md', 'page': 2}]


### Final Mini Exercise Solution

In [ ]:
# Final Mini Exercise Solution
chunks = [
    {"chunk_id": "arabic-book-001", "text": "إِنَّ الذكاءَ الاصطناعيَّ يساعدُ في تحليلِ النصوصِ.", "source": "arabic_ai_book.md", "page": 1},
    {"chunk_id": "arabic-book-002", "text": "قواعدُ البياناتِ المتجهيةُ مفيدةٌ في البحثِ الدلالي.", "source": "arabic_ai_book.md", "page": 2},
]

def prepare_arabic_chunks_for_retrieval(chunks):
    prepared_records = []
    for chunk in chunks:
        raw_text = chunk["text"]
        normalized_text = clean_arabic_text(raw_text)
        prepared_records.append({
            "chunk_id": chunk["chunk_id"],
            "raw_text": raw_text,
            "normalized_text": normalized_text,
            "source": chunk["source"],
            "page": chunk["page"],
            "text_length": len(normalized_text),
            "ready_for_embedding": len(normalized_text) > 20,
        })
    return prepared_records

prepared = prepare_arabic_chunks_for_retrieval(chunks)
pd.DataFrame(prepared)

## Key Takeaways

- AraBERT is an important Arabic BERT-style model for language understanding.
- Similar Arabic models include CAMeLBERT, MARBERT, ARBERT, and AraELECTRA.
- Modern Arabic RAG often needs embedding models, not only masked-language models.
- BGE-M3 and Arabic-tuned embedding models are useful candidates for retrieval.
- Distinguish between encoder, embedding model, reranker, and generator.
- Arabic preprocessing matters: diacritics, Alef forms, Ya forms, tatweel, and whitespace.
- Store both raw text and normalized text in production systems.
- Use the same embedding model for documents and queries.
- Evaluate models on your actual Arabic domain.
- Arabic NLP modules support ingestion, retrieval, ranking, evaluation, and metadata enrichment.